## 1. Install Library

In [ ]:
!pip install transformers datasets
# !pip install rouge-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.4/485.4 kB 37.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 14.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.8/194.8 kB 19.8 MB/s eta 0:00:00


## 2. Persiapan Data

In [ ]:
import pandas as pd
from datasets import Dataset

# Contoh DataFrame
data = {
    "product": [
        "Smartphone dengan kamera 108MP",
        "Baju kaos katun warna hitam",
        "Buku novel karya Tere Liye",
        "Laptop gaming dengan RAM 16GB",
        "Sepatu lari Nike",
        "Buku resep masakan"
    ],
    "category": [
        "Electronics",
        "Fashion",
        "Books",
        "Electronics",
        "Fashion",
        "Books"
    ]
}

df = pd.DataFrame(data)

# Konversi DataFrame ke Dataset Hugging Face
dataset = Dataset.from_pandas(df)

# Pisahkan dataset menjadi train dan test
dataset = dataset.train_test_split(test_size=0.2)
dataset

DatasetDict({
    train: Dataset({
        features: ['product', 'category'],
        num_rows: 4
    })
    test: Dataset({
        features: ['product', 'category'],
        num_rows: 2
    })
})

## 3. Tokenisasi Data

Kita akan menggunakan tokenizer dari model distilbert-base-uncased

In [ ]:
from transformers import AutoTokenizer

# Pilih model dan tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Fungsi untuk tokenisasi data
def tokenize_function(examples):
    return tokenizer(examples["product"], padding="max_length", truncation=True)

# Tokenisasi dataset
tokenized_datasets = dataset.map(tokenize_function, batched=True)

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

## 4. Persiapan Model

Kita akan menambahkan lapisan klasifikasi di atas model distilbert-base-uncased

In [ ]:
from transformers import AutoModelForSequenceClassification

# Pilih model untuk klasifikasi
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(df["category"].unique())  # Jumlah kategori unik
)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 5. Fine-Tuning Model

Kita akan menggunakan Trainer dari Hugging Face untuk melakukan fine-tuning.

In [ ]:
from transformers import Trainer, TrainingArguments

# Persiapan label
label_list = df["category"].unique().tolist()
label_to_id = {label: idx for idx, label in enumerate(label_list)}

# Fungsi untuk mengonversi label ke ID
def map_labels(examples):
    examples["label"] = label_to_id[examples["category"]]
    return examples

# Terapkan fungsi ke dataset
tokenized_datasets = tokenized_datasets.map(map_labels)

# Argumen pelatihan
training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    logging_steps=10,
    report_to="none",
)

# Inisialisasi Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
)

# Mulai fine-tuning
trainer.train()

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

Map:   0%|          | 0/2 [00:00<?, ? examples/s]

/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
<ipython-input-5-d9cbf10a7223>:30: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss
1,No log,1.144968
2,No log,1.144598
3,No log,1.143328


TrainOutput(global_step=3, training_loss=1.0703299045562744, metrics={'train_runtime': 3.2412, 'train_samples_per_second': 3.702, 'train_steps_per_second': 0.926, 'total_flos': 1589637132288.0, 'train_loss': 1.0703299045562744, 'epoch': 3.0})

## 6. Evaluasi Model

In [ ]:
# Evaluasi model
eval_results = trainer.evaluate()
print(f"Evaluasi hasil: {eval_results}")

Evaluasi hasil: {'eval_loss': 1.1433281898498535, 'eval_runtime': 0.0481, 'eval_samples_per_second': 41.584, 'eval_steps_per_second': 20.792, 'epoch': 3.0}


## 7. Menggunakan Model untuk Prediksi

In [ ]:
import torch

# Pastikan model berada di perangkat yang benar (GPU atau CPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)  # Pindahkan model ke GPU jika tersedia

# Fungsi untuk prediksi
def predict_category(product_text):
    # Tokenisasi input
    inputs = tokenizer(product_text, return_tensors="pt", padding=True, truncation=True)

    # Pindahkan input ke perangkat yang sama dengan model
    inputs = {key: value.to(device) for key, value in inputs.items()}

    # Lakukan prediksi
    with torch.no_grad():  # Nonaktifkan perhitungan gradien
        outputs = model(**inputs)

    # Ambil prediksi kelas
    predicted_class_id = outputs.logits.argmax().item()
    return label_list[predicted_class_id]

# Test prediksi
test_products = [
    "Smartphone dengan baterai 5000mAh",
    "Jaket kulit pria",
    "Buku panduan programming Python"
]

for product in test_products:
    predicted_category = predict_category(product)
    print(f"Product: {product}")
    print(f"Predicted Category: {predicted_category}")
    print("-" * 50)

Product: Smartphone dengan baterai 5000mAh
Predicted Category: Electronics
--------------------------------------------------
Product: Jaket kulit pria
Predicted Category: Electronics
--------------------------------------------------
Product: Buku panduan programming Python
Predicted Category: Electronics
--------------------------------------------------
